# 15b · GSE232381 · bulk_RNA_seq · scoring these samples on the GSE65391 modules

Reads this study's `expression.rds` and the GSE65391 end product `module_loadings.csv` (for each module:
its genes and their weights; no GSE65391 expression values). Writes `projected_scores.rds` in
`data/run_artifacts/GSE232381/`.

**What is transferred.** The model fitted on the GSE65391 first visits (notebook 05b): for each module,
the genes and their weights (u₁ / d₁ of the module's first principal component). Scoring a new data set
with a model learned elsewhere is transfer learning by projection (Stein-O'Brien GL et al. Decomposing
cell identity for transfer learning across cellular measurements, platforms, tissues, and species.
*Cell Syst* 2019;8:395–411, doi:10.1016/j.cels.2019.04.004; Sharma G et al. projectR. *Bioinformatics*
2020;36:3592–3593, doi:10.1093/bioinformatics/btaa183).

**Scaling is local.** GSE65391 standardised each gene with its own first-visit mean and SD. These are
array intensities; here the values are RNA sequencing log counts on another scale, so the array's means
and SDs cannot be applied. Each gene is standardised within this study, over its 16 samples, and then
weighted with the array weights:

  score = Σ over the module's genes measured here of z × weight

**Coverage.** Genes that this study does not measure are left out, and the share of each module's genes
used is reported.

**Check.** For each module, the projected score against the eigengene recomputed from this study's own
data (notebook 16's "array module" family): Pearson r. A high r means the array's weighting and a local
refit agree on the module's axis in these samples.

In [1]:
source("../src/paths.R")
x  <- readRDS(art("GSE232381", "expression.rds"))
ld <- read.csv(art("GSE65391", "module_loadings.csv"))
E  <- x$E[, x$meta$sample]
Z  <- (E - rowMeans(E)) / apply(E, 1, sd)                 # genes x samples, standardised in this study
Z  <- Z[apply(E, 1, sd) > 0, ]
cov <- do.call(rbind, lapply(split(ld, ld$module), function(d)
  data.frame(module = d$module[1], genes_in_array_module = nrow(d), genes_here = sum(d$gene %in% rownames(Z)))))
cov$share_used <- round(cov$genes_here / cov$genes_in_array_module, 3)
cov[order(cov$share_used), ]

,module,genes_in_array_module,genes_here,share_used
,<chr>,<int>,<int>,<dbl>
black,black,250,97,0.388
yellow,yellow,416,216,0.519
green,green,301,161,0.535
salmon,salmon,55,32,0.582
greenyellow,greenyellow,84,63,0.750
magenta,magenta,103,80,0.777
brown,brown,426,359,0.843
midnightblue,midnightblue,44,39,0.886
red,red,268,240,0.896


**Result.** Coverage ranges from 39% (black: many of its array probes are ribosomal pseudogenes and LOC
entries that the RNA sequencing counts do not include) to 100% (tan); 12 of 16 modules have 75% or more.

In [2]:
scores <- sapply(split(ld, ld$module), function(d) {
  d <- d[d$gene %in% rownames(Z), ]
  colSums(Z[d$gene, , drop = FALSE] * d$weight)
})
dim(scores)

[1] 16 16

**Check.** Projected score against the locally recomputed eigengene of the same gene list.

In [3]:
suppressMessages(library(WGCNA))
ld_here <- ld[ld$gene %in% rownames(Z), ]
ME_local <- moduleEigengenes(t(E[ld_here$gene, ]), colors = ld_here$module)$eigengenes
check <- data.frame(module = colnames(scores),
                    r_projected_vs_local = sapply(colnames(scores), function(mo) cor(scores[, mo], ME_local[, paste0("ME", mo)])),
                    row.names = NULL)
check$r_projected_vs_local <- round(check$r_projected_vs_local, 3)
check[order(check$r_projected_vs_local), ]

,module,r_projected_vs_local
,<chr>,<dbl>
15,turquoise,0.437
5,green,0.789
12,red,0.864
13,salmon,0.871
16,yellow,0.976
10,pink,0.987
1,black,0.998
6,greenyellow,0.998
11,purple,0.998


**Result.** For 12 of 16 modules the projected score and the local refit agree closely (r = 0.976 to
1.000), including pink (interferon, 0.987), greenyellow (plasma cell, 0.998) and lightcyan (neutrophil
granule, 0.999). Four agree less: turquoise (0.437), green (0.789), red (0.864) and salmon (0.871).
Turquoise, green and salmon are the modules notebook 15 found not preserved in this study.

In [4]:
saveRDS(list(scores = scores, coverage = cov, check = check), art("GSE232381", "projected_scores.rds"))